# 03 · Train ResNet-50

Full fine-tuning of **ResNet-50** (ImageNet-V2 pretrained) for 16-class classification.

**Checklist:**
- [ ] Notebook 02 complete — `/content/data` has train/val/test  OR  Drive split exists
- [ ] Runtime → Runtime type → **A100 GPU**

In [1]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────
import os, sys

GITHUB_USER = 'musarashid49'
REPO_NAME   = 'Image-Classification-with-CNN'
REPO_DIR    = f'/content/{REPO_NAME}'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ NOT FOUND')
print('CUDA:', torch.version.cuda)

Cloning into '/content/Image-Classification-with-CNN'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 48 (delta 11), reused 43 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 36.93 KiB | 18.47 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
CUDA: 12.8


In [5]:
# ── Cell 2: Paths ─────────────────────────────────────────────────────
LOCAL_DATA_DIR   = '/content/data'
DRIVE_SPLIT_PATH = '/content/drive/MyDrive/dataset_resplit'  # ← changed
DRIVE_RESULTS    = '/content/drive/MyDrive/pk_politicians_results'

In [6]:
import shutil, os, time

DRIVE_SPLIT_PATH = '/content/drive/MyDrive/dataset_resplit'
LOCAL_DATA_DIR = '/content/data'

if not os.path.isdir(LOCAL_DATA_DIR) or len(os.listdir(LOCAL_DATA_DIR)) == 0:
    print("Copying dataset from Drive → SSD...")
    t0 = time.time()
    shutil.rmtree(LOCAL_DATA_DIR, ignore_errors=True)
    shutil.copytree(DRIVE_SPLIT_PATH, LOCAL_DATA_DIR)
    print(f"✓ Done in {time.time()-t0:.1f}s")
else:
    print("✓ Data already on SSD — skipping copy")

from src.dataset import get_dataloaders
loaders = get_dataloaders(
    train_dir = os.path.join(LOCAL_DATA_DIR, 'train'),
    val_dir   = os.path.join(LOCAL_DATA_DIR, 'val'),
    test_dir  = os.path.join(LOCAL_DATA_DIR, 'test'),
)

Copying dataset from Drive → SSD...
✓ Done in 982.3s

── DataLoader Summary ──────────────────────────────────────
  train     1064 images  |    33 batches  |  classes: 16
  val        205 images  |     7 batches  |  classes: 16
  test       159 images  |     5 batches  |  classes: 16

  Class-to-index mapping:
    [ 0]  ahmed_sharif_chaudhry
    [ 1]  altaf_hussain
    [ 2]  asfandyar_wali
    [ 3]  asif_ali_zardari
    [ 4]  bilawal_bhutto
    [ 5]  chaudhry_nisar
    [ 6]  fazlur_rehman
    [ 7]  imran_khan
    [ 8]  maryam_nawaz
    [ 9]  nawaz_sharif
    [10]  pervez_khattak
    [11]  pervez_musharraf
    [12]  rana_sanaullah
    [13]  shah_mehmood_qureshi
    [14]  shehbaz_sharif
    [15]  sirajul_haq


In [11]:
# ── Cell 4: Build ResNet-50 ────────────────────────────────────────────
from src.models import build_model

model = build_model(
    model_name  = 'resnet50',
    num_classes = 16,
    dropout     = 0.4,
    freeze_base = False,   # full fine-tuning
)

  [resnet50]  trainable params: 23,540,816 / 23,540,816


In [14]:
# ── Cell 5: Train ─────────────────────────────────────────────────────
from src.train import Trainer

trainer = Trainer(
    model      = model,
    loaders    = loaders,
    model_name = 'resnet50',
    config_overrides = {
        'num_epochs':    30,
        'learning_rate': 1e-4,
        'weight_decay':  1e-4,
    }
)
history = trainer.run()

TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

In [ ]:
# ── Cell 6: Training curves ────────────────────────────────────────────
from src.evaluate import plot_training_curves
plot_training_curves(history, model_name='resnet50').show()

In [ ]:
# ── Cell 7: Test set evaluation ───────────────────────────────────────
import torch
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_misclassified

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = evaluate_model(model, loaders['test'], device, model_name='resnet50')

plot_confusion_matrix(results['y_true'], results['y_pred'], model_name='resnet50').show()
fig = plot_misclassified(results['images'], results['y_true'], results['y_pred'], model_name='resnet50')
if fig: fig.show()

In [ ]:
# ── Cell 8: Log experiment ────────────────────────────────────────────
from src.utils import ExperimentLogger
logger = ExperimentLogger()
logger.log(
    model_name    = 'resnet50',
    test_accuracy = round(results['accuracy'], 4),
    macro_f1      = round(results['report']['macro avg']['f1-score'], 4),
    epochs_run    = len(history['train_loss']),
    lr=1e-4, batch_size=32,
    notes = 'Full fine-tuning, dropout=0.4, early stop patience=8',
)

In [ ]:
# ── Cell 9: Save results to Drive ─────────────────────────────────────
from src.utils import save_results_to_drive
save_results_to_drive('results', DRIVE_RESULTS)
print('✓ Saved. Next → 04_train_efficientnet.ipynb')